In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/kushagragupta28xoxo/traffic-light-violation/test_images/4.png
/kaggle/input/datasets/kushagragupta28xoxo/traffic-light-violation/test_images/1.png
/kaggle/input/datasets/kushagragupta28xoxo/traffic-light-violation/test_images/2.png
/kaggle/input/datasets/kushagragupta28xoxo/traffic-light-violation/test_images/3.png
/kaggle/input/datasets/kushagragupta28xoxo/traffic-video/Test video 1.mp4


In [2]:
import cv2

video_path = "/kaggle/input/datasets/kushagragupta28xoxo/traffic-video/Test video 1.mp4"

cap = cv2.VideoCapture(video_path)

print("Opened:", cap.isOpened())
print("FPS:", cap.get(cv2.CAP_PROP_FPS))
print("Frames:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
print("Width:", int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)))
print("Height:", int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))

cap.release()

Opened: True
FPS: 30.0
Frames: 305
Width: 1280
Height: 720


In [3]:
# =============================================================================
# CELL 1 — Install dependencies (FIXED)
# =============================================================================

import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q",
                           "--no-deps" if "boxmot" in pkg else ""],
                          )

# Remove empty string arg issue — cleaner version:
def install(pkg, extra_args=None):
    cmd = [sys.executable, "-m", "pip", "install", pkg, "-q"]
    if extra_args:
        cmd += extra_args
    subprocess.check_call(cmd)

install("ultralytics")                          # YOLOv8
install("boxmot")                               # ByteTrack, StrongSORT, etc.
install("easyocr")                              # License plate OCR
install("supervision")                          # Annotation helpers
install("lap", ["--no-build-isolation"])        # Required by boxmot tracker

print("✅ All packages installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 95.4 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 kB 25.2 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
sigstore 4.2.0 requires rich<15,>=13, but you have rich 15.0.0 which is incompatible.
bigframes 2.39.0 requires rich<14,>=12.4.4, but you have rich 15.0.0 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.6 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you have numba-cuda 0.30.2 which is incompatible.
pyiceberg 0.11.1 requires rich

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.6/273.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 21.1 MB/s eta 0:00:00
✅ All packages installed


In [4]:
# =============================================================================
# CELL 2 — Imports + Global Config (Supervision ByteTrack Version)
# =============================================================================

import cv2
import json
import time
import easyocr
import numpy as np
import supervision as sv

from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, asdict
from collections import defaultdict
from ultralytics import YOLO

# =============================================================================
# Output directories
# =============================================================================

OUT_DIR = Path("/kaggle/working/violations")
FRAMES_DIR = OUT_DIR / "frames"
VIDEO_DIR = OUT_DIR / "annotated_videos"
REPORT_DIR = OUT_DIR / "reports"

for d in [FRAMES_DIR, VIDEO_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# =============================================================================
# Detection thresholds
# =============================================================================

VEHICLE_CONF = 0.40
TL_CONF = 0.35
IOU_THRESH = 0.45

# =============================================================================
# Classes of interest (COCO)
# =============================================================================

VEHICLE_CLASSES = {
    2: "car",
    3: "motorcycle",
    5: "bus",
    7: "truck"
}

TL_CLASS_ID = 9  # COCO traffic light class

# =============================================================================
# HSV ranges for traffic-light color detection
# =============================================================================

HSV_RED_1 = (
    np.array([0, 120, 70]),
    np.array([10, 255, 255])
)

HSV_RED_2 = (
    np.array([170, 120, 70]),
    np.array([180, 255, 255])
)

HSV_GREEN = (
    np.array([40, 50, 50]),
    np.array([90, 255, 255])
)

HSV_YELLOW = (
    np.array([15, 100, 100]),
    np.array([40, 255, 255])
)

# =============================================================================
# ByteTrack parameters (Supervision)
# =============================================================================

TRACK_ACTIVATION_THRESHOLD = 0.45
LOST_TRACK_BUFFER = 30
MINIMUM_MATCHING_THRESHOLD = 0.85
FRAME_RATE = 30

# =============================================================================
# Violation metadata structure
# =============================================================================

@dataclass
class Violation:
    violation_id: str
    timestamp: str
    frame_number: int
    track_id: int
    vehicle_class: str
    tl_state: str
    tl_confidence: float
    vehicle_conf: float
    bbox_xyxy: list
    license_plate: str = "undetected"
    frame_saved_at: str = ""
    notes: str = ""

print("✅ Config ready")
print("📁 Output directory:", OUT_DIR)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Config ready
📁 Output directory: /kaggle/working/violations


In [5]:
# =============================================================================
# CELL 3 — Core helper functions
# =============================================================================
 
# ── 3-A  Image preprocessing (CLAHE + denoising) ─────────────────────────────
 
def preprocess_frame(frame: np.ndarray) -> np.ndarray:
    """
    Apply CLAHE on L-channel for low-light / high-contrast scenes,
    then fast non-local means denoising for rain / sensor noise.
    Returns a BGR frame ready for YOLO inference.
    """
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    l = clahe.apply(l)
    lab = cv2.merge([l, a, b])
    enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    denoised = cv2.fastNlMeansDenoisingColored(enhanced, None, 5, 5, 7, 21)
    return denoised
 
 
# ── 3-B  Traffic-light color via HSV ─────────────────────────────────────────
 
def classify_tl_color(frame: np.ndarray, bbox_xyxy) -> tuple[str, float]:
    """
    Crop the traffic-light bounding box, use HSV masking to detect red /
    green / yellow.  Returns (color_str, pseudo_confidence).
    """
    x1, y1, x2, y2 = map(int, bbox_xyxy)
    # Focus on the top 60% of the TL box (lamps are usually top)
    h = y2 - y1
    crop = frame[y1 : y1 + int(h * 0.6), x1:x2]
    if crop.size == 0:
        return "unknown", 0.0
 
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
 
    red_mask = (cv2.inRange(hsv, HSV_RED_1[0], HSV_RED_1[1]) |
                cv2.inRange(hsv, HSV_RED_2[0], HSV_RED_2[1]))
    green_mask  = cv2.inRange(hsv, HSV_GREEN[0],  HSV_GREEN[1])
    yellow_mask = cv2.inRange(hsv, HSV_YELLOW[0], HSV_YELLOW[1])
 
    counts = {
        "red":    int(cv2.countNonZero(red_mask)),
        "green":  int(cv2.countNonZero(green_mask)),
        "yellow": int(cv2.countNonZero(yellow_mask)),
    }
    total = sum(counts.values()) or 1
    best  = max(counts, key=counts.get)
    conf  = round(counts[best] / total, 3)
    return (best, conf) if counts[best] > 30 else ("unknown", 0.0)
 
 
# ── 3-C  Stop-line ROI helper ─────────────────────────────────────────────────
 
def get_default_rois(frame_h: int, frame_w: int) -> dict:
    """
    Returns two ROI polygons as numpy arrays.
    ROI-A = strip just before the stop line (vehicles waiting).
    ROI-B = strip just after  the stop line (violation zone).
 
    Adjust the y-fractions below to match your actual camera angle.
    For easy calibration these are printed as a guide; see Cell 4 for
    the interactive calibration helper.
    """
    roi_a_y1 = int(frame_h * 0.55)   # 55% down the frame
    roi_a_y2 = int(frame_h * 0.65)   # 65% down
    roi_b_y1 = int(frame_h * 0.65)   # immediately after stop line
    roi_b_y2 = int(frame_h * 0.78)   # 78% down
 
    roi_a = np.array([[0, roi_a_y1], [frame_w, roi_a_y1],
                       [frame_w, roi_a_y2], [0, roi_a_y2]])
    roi_b = np.array([[0, roi_b_y1], [frame_w, roi_b_y1],
                       [frame_w, roi_b_y2], [0, roi_b_y2]])
    return {"A": roi_a, "B": roi_b}
 
 
def point_in_roi(cx: int, cy: int, roi_pts: np.ndarray) -> bool:
    return cv2.pointPolygonTest(roi_pts, (float(cx), float(cy)), False) >= 0
 
 
# ── 3-D  License plate OCR ────────────────────────────────────────────────────
 
_ocr_reader = None
 
def get_ocr():
    global _ocr_reader
    if _ocr_reader is None:
        _ocr_reader = easyocr.Reader(["en"], gpu=True, verbose=False)
    return _ocr_reader
 
def read_plate(frame: np.ndarray, vehicle_bbox) -> str:
    """
    Crop the bottom third of the vehicle bbox (plates are usually there),
    run EasyOCR, return the best alphanumeric string.
    """
    x1, y1, x2, y2 = map(int, vehicle_bbox)
    h = y2 - y1
    plate_crop = frame[y1 + int(h * 0.65) : y2, x1:x2]
    if plate_crop.size == 0:
        return "undetected"
    # Upscale for OCR accuracy
    plate_crop = cv2.resize(plate_crop, None, fx=2, fy=2,
                            interpolation=cv2.INTER_CUBIC)
    results = get_ocr().readtext(plate_crop, allowlist="ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789")
    if not results:
        return "undetected"
    best = max(results, key=lambda r: r[2])   # highest confidence
    text = best[1].strip().upper().replace(" ", "")
    return text if len(text) >= 3 else "undetected"
 
 
# ── 3-E  Annotation helpers ───────────────────────────────────────────────────
 
def draw_rois(frame, rois):
    overlay = frame.copy()
    cv2.fillPoly(overlay, [rois["A"]], (0, 255, 255))   # yellow tint
    cv2.fillPoly(overlay, [rois["B"]], (0, 0, 255))      # red tint
    cv2.addWeighted(overlay, 0.18, frame, 0.82, 0, frame)
    cv2.polylines(frame, [rois["A"]], True, (0, 220, 220), 2)
    cv2.polylines(frame, [rois["B"]], True, (0, 0, 220), 2)
    h, w = frame.shape[:2]
    cv2.putText(frame, "ROI-A (waiting zone)", (10, rois["A"][0][1] - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 220, 220), 1)
    cv2.putText(frame, "ROI-B (violation zone)", (10, rois["B"][0][1] - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 220), 1)
    return frame
 
 
TL_COLOR_MAP = {"red": (0, 0, 255), "green": (0, 200, 0),
                "yellow": (0, 200, 200), "unknown": (128, 128, 128)}
 
def draw_vehicle(frame, bbox, track_id, v_class, is_violator=False):
    x1, y1, x2, y2 = map(int, bbox)
    color = (0, 0, 255) if is_violator else (0, 200, 100)
    thickness = 3 if is_violator else 2
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    label = f"#{track_id} {v_class}"
    if is_violator:
        label += " ⚠ VIOLATION"
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
    cv2.rectangle(frame, (x1, y1 - th - 6), (x1 + tw + 4, y1), color, -1)
    cv2.putText(frame, label, (x1 + 2, y1 - 4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
    return frame
 
 
def draw_tl_state(frame, tl_color, tl_conf):
    color = TL_COLOR_MAP.get(tl_color, (128, 128, 128))
    cv2.circle(frame, (30, 30), 18, color, -1)
    cv2.circle(frame, (30, 30), 18, (255, 255, 255), 2)
    label = f"TL: {tl_color.upper()}  ({tl_conf:.0%})"
    cv2.putText(frame, label, (55, 38),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)
    return frame
 
 
def draw_hud(frame, frame_no, violation_count, fps_val):
    h, w = frame.shape[:2]
    ts = datetime.now().strftime("%Y-%m-%d  %H:%M:%S")
    cv2.putText(frame, f"Frame: {frame_no}   |   Violations: {violation_count}   |   {fps_val:.1f} FPS",
                (10, h - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200, 200, 200), 1)
    cv2.putText(frame, ts, (w - 220, h - 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180, 180, 180), 1)
    return frame
 
 
def save_violation_frame(frame, violation: Violation):
    fname = FRAMES_DIR / f"violation_{violation.violation_id}.jpg"
    cv2.imwrite(str(fname), frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
    return str(fname)
 
 
print("✅ All helper functions defined")

✅ All helper functions defined


In [23]:
# =============================================================================
# CELL 4 — Model loading + ROI calibration preview
# =============================================================================

print("Loading YOLOv8x … (downloads ~130 MB on first run)")
model = YOLO("yolov8x.pt")
model.to("cuda")

print(f"✅ Model loaded on {next(model.model.parameters()).device}")

# -----------------------------------------------------------------------------
# SAMPLE IMAGE FOR ROI CALIBRATION
# -----------------------------------------------------------------------------

SAMPLE_INPUT = "/kaggle/input/datasets/kushagragupta28xoxo/imageeeee/test.png"

# -----------------------------------------------------------------------------
# ROI CONFIGURATION
# -----------------------------------------------------------------------------

ROI_A_FRAC = (0.30, 0.45)
ROI_B_FRAC = (0.45, 0.60)

ROAD_LEFT_FRAC  = 0.20
ROAD_RIGHT_FRAC = 0.80


# -----------------------------------------------------------------------------
# ROI CREATION
# -----------------------------------------------------------------------------

def get_default_rois(h, w):

    # ==========================
    # GREEN ROI (WAITING ZONE)
    # ==========================
    roi_a = np.array([
        [0, 520],     # top-left
        [1020, 610],    # top-right
        [1020, 610],   # bottom-right
        [0, 520],      # bottom-left
    ], dtype=np.int32)

    # ==========================
    # RED ROI (VIOLATION ZONE)
    # ==========================
    roi_b = np.array([
        [250, 80],     # top-left
        [1050, 150],   # top-right
        [1030, 360],   # bottom-right
        [220, 260],    # bottom-left
    ], dtype=np.int32)

    return {
        "A": roi_a,
        "B": roi_b
    }

# -----------------------------------------------------------------------------
# ROI PREVIEW
# -----------------------------------------------------------------------------

def preview_rois(image_path):

    img = cv2.imread(image_path)

    if img is None:
        print(f"❌ Could not read image: {image_path}")
        return

    h, w = img.shape[:2]

    rois = get_default_rois(h, w)

    vis = img.copy()

    # ROI A (GREEN)
    cv2.polylines(
        vis,
        [rois["A"].astype(np.int32)],
        True,
        (0, 255, 0),
        4
    )

    # ROI B (RED)
    cv2.polylines(
        vis,
        [rois["B"].astype(np.int32)],
        True,
        (0, 0, 255),
        4
    )

    cv2.putText(
        vis,
        "ROI A",
        (
            int(rois["A"][0][0]),
            int(rois["A"][0][1] - 10)
        ),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0,255,0),
        3
    )

    cv2.putText(
        vis,
        "ROI B",
        (
            int(rois["B"][0][0]),
            int(rois["B"][0][1] - 10)
        ),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0,0,255),
        3
    )

    out_path = str(OUT_DIR / "roi_preview.jpg")

    cv2.imwrite(out_path, vis)

    print("\n✅ ROI preview saved")
    print(out_path)

    print("\nFrame size:")
    print(f"Width  = {w}")
    print(f"Height = {h}")

    print("\nROI A coordinates:")
    print(rois["A"])

    print("\nROI B coordinates:")
    print(rois["B"])

    print("\nROI settings:")
    print(f"ROI_A_FRAC = {ROI_A_FRAC}")
    print(f"ROI_B_FRAC = {ROI_B_FRAC}")
    print(f"ROAD_LEFT_FRAC  = {ROAD_LEFT_FRAC}")
    print(f"ROAD_RIGHT_FRAC = {ROAD_RIGHT_FRAC}")

    return vis


# -----------------------------------------------------------------------------
# RUN PREVIEW
# -----------------------------------------------------------------------------

preview_rois(SAMPLE_INPUT)

Loading YOLOv8x … (downloads ~130 MB on first run)
✅ Model loaded on cuda:0

✅ ROI preview saved
/kaggle/working/violations/roi_preview.jpg

Frame size:
Width  = 1251
Height = 821

ROI A coordinates:
[[  20  260]
 [ 860  360]
 [1020  610]
 [   0  520]]

ROI B coordinates:
[[ 250   80]
 [1050  150]
 [1030  360]
 [ 220  260]]

ROI settings:
ROI_A_FRAC = (0.3, 0.45)
ROI_B_FRAC = (0.45, 0.6)
ROAD_LEFT_FRAC  = 0.2
ROAD_RIGHT_FRAC = 0.8


array([[[ 0,  0,  0],
        [ 0,  0,  0],
        [ 0,  0,  0],
        ...,
        [ 0,  0,  0],
        [ 0,  0,  0],
        [ 0,  0,  0]],

       [[ 0,  0,  0],
        [ 0,  0,  0],
        [ 0,  0,  0],
        ...,
        [ 0,  0,  0],
        [ 0,  0,  0],
        [ 0,  0,  0]],

       [[ 0,  0,  0],
        [ 0,  0,  0],
        [ 0,  0,  0],
        ...,
        [ 0,  0,  0],
        [ 0,  0,  0],
        [ 0,  0,  0]],

       ...,

       [[53, 51, 51],
        [53, 51, 51],
        [53, 51, 51],
        ...,
        [29, 36, 33],
        [33, 40, 37],
        [32, 38, 36]],

       [[53, 51, 51],
        [53, 51, 51],
        [53, 51, 51],
        ...,
        [28, 35, 32],
        [33, 40, 37],
        [32, 39, 36]],

       [[20, 20, 20],
        [20, 20, 20],
        [20, 20, 20],
        ...,
        [20, 20, 20],
        [20, 20, 20],
        [20, 20, 20]]], shape=(821, 1251, 3), dtype=uint8)

In [7]:
print(sv.__version__)

tracker = sv.ByteTrack()

print(hasattr(tracker, "update_with_detections"))

0.29.0.post0
True


In [18]:
# =============================================================================
# CELL 5 — Main detection engine
# =============================================================================
def _video_iter(cap, max_frames=None):
    frame_no = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        yield frame_no, frame
        frame_no += 1

        if max_frames is not None and frame_no >= max_frames:
            break


def _image_iter(paths):
    for i, p in enumerate(paths):
        img = cv2.imread(str(p))

        if img is None:
            continue

        yield i, img


def run_violation_detection(
    source: str,
    roi_a_frac=(0.60, 0.75)
    roi_b_frac=(0.46, 0.60),
    save_video: bool  = True,
    max_frames: int   = None,            # None = process whole video
) -> list[Violation]:
    """
    Main pipeline. Works on both video files (.mp4/.avi) and
    image folders (pass a glob string like "/path/to/imgs/*.jpg").
 
    Returns a list of Violation dataclass instances.
    """
 
    # ── Resolve source ──────────────────────────────────────────────────────
    source_path = Path(source)
    is_video    = source_path.suffix.lower() in {".mp4", ".avi", ".mov", ".mkv"}
    is_image    = source_path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
    is_glob     = "*" in source
 
    if is_video:
        cap    = cv2.VideoCapture(source)
        fps    = cap.get(cv2.CAP_PROP_FPS) or 25
        fw     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        fh     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        frames_iter = _video_iter(cap, max_frames)
        print(f"📹 Video  {fw}×{fh}  {fps:.1f} fps")
    elif is_glob or source_path.is_dir():
        import glob
        pattern = source if is_glob else str(source_path / "*.jpg")
        paths   = sorted(glob.glob(pattern))
        cap, fps, fw, fh = None, 1, None, None
        frames_iter = _image_iter(paths)
        print(f"🖼  Images  {len(paths)} files")
    elif is_image:
        cap, fps, fw, fh = None, 1, None, None
        frames_iter = _image_iter([source])
        print(f"🖼  Single image: {source}")
    else:
        raise ValueError(f"Unknown source type: {source}")
 
    # ── Setup ────────────────────────────────────────────────────────────────
    tracker = sv.ByteTrack(
    track_activation_threshold=0.45,
    lost_track_buffer=30,
    minimum_matching_threshold=0.85,
    frame_rate=int(fps),
    )

    
    violations   : list[Violation] = []
    all_metadata : dict            = {}    # track_id → partial violation data
    in_roi_a     : set[int]        = set() # track IDs seen in ROI-A during red
    violated     : set[int]        = set() # confirmed violators this red phase
    tl_state     = "unknown"
    tl_conf      = 0.0
    frame_no     = 0
    vio_count    = 0
    rois         = None                    # set on first frame
    writer       = None
    t0           = time.time()
 
    def _get_rois(h, w):
        a1, a2 = int(h * roi_a_frac[0]), int(h * roi_a_frac[1])
        b1, b2 = int(h * roi_b_frac[0]), int(h * roi_b_frac[1])
        roi_a = np.array([[0,a1],[w,a1],[w,a2],[0,a2]])
        roi_b = np.array([[0,b1],[w,b1],[w,b2],[0,b2]])
        return {"A": roi_a, "B": roi_b}
 
    # ── Frame loop ───────────────────────────────────────────────────────────
    for frame_no, frame in frames_iter:
 
        h, w = frame.shape[:2]
        if rois is None:
            rois = _get_rois(h, w)
            if save_video and is_video and fw:
                vout_path = str(VIDEO_DIR / f"annotated_{source_path.stem}.mp4")
                fourcc    = cv2.VideoWriter_fourcc(*"mp4v")
                writer    = cv2.VideoWriter(vout_path, fourcc, fps, (fw, fh))
 
        # 1. Preprocess
        proc = preprocess_frame(frame)
 
        # 2. YOLO inference (detect vehicles + traffic lights simultaneously)
        # ==========================================================
        # VEHICLE DETECTION ON FULL FRAME
        # ==========================================================
        results = model(
            proc,
            conf=min(VEHICLE_CONF, TL_CONF),
            iou=IOU_THRESH,
            verbose=False,
            device="cuda",
        )[0]
        
        # ==========================================================
        # TRAFFIC LIGHT DETECTION ON TOP HALF ONLY
        # ==========================================================
        top_half = frame[: h // 2, :]
        
        tl_results = model(
            top_half,
            conf=0.10,          # lower threshold
            iou=0.45,
            verbose=False,
            device="cuda",
        )[0]
 
        boxes  = results.boxes
        dets   = boxes.xyxy.cpu().numpy()    # [N, 4]
        confs  = boxes.conf.cpu().numpy()    # [N]
        cls_ids= boxes.cls.cpu().numpy().astype(int)  # [N]
 
        # 3. Split detections
        # ===========================
        # VEHICLES FROM FULL FRAME
        # ===========================
        vehicle_mask = np.array([c in VEHICLE_CLASSES for c in cls_ids])
        
        veh_dets  = dets[vehicle_mask]
        veh_confs = confs[vehicle_mask]
        veh_cls   = cls_ids[vehicle_mask]
        
        # ===========================
        # TRAFFIC LIGHTS FROM TOP HALF
        # ===========================
        tl_boxes = tl_results.boxes
        
        if len(tl_boxes) > 0:
        
            tl_dets = tl_boxes.xyxy.cpu().numpy()
            tl_confs = tl_boxes.conf.cpu().numpy()
            tl_cls = tl_boxes.cls.cpu().numpy().astype(int)
        
            tl_mask = tl_cls == TL_CLASS_ID
        
            tl_dets = tl_dets[tl_mask]
            tl_confs = tl_confs[tl_mask]
        
        else:
        
            tl_dets = np.empty((0, 4))
            tl_confs = np.array([])
 
        # 4. Traffic-light color classification


        print(
                f"Frame {frame_no}: "
                f"Traffic lights found = {len(tl_dets)}"
            )

        # 4. Traffic-light color classification
        if len(tl_dets) > 0:
            best_tl_idx = int(np.argmax(tl_confs))
            tl_state, tl_conf = classify_tl_color(frame, tl_dets[best_tl_idx])
        
            # Fallback: if color cannot be determined, assume RED
            if tl_state == "unknown":
                tl_state = "red"
                tl_conf = 0.50
        else:
            # No traffic light detected at all
            tl_state = "red"
            tl_conf = 0.50
 
        # 5. ByteTrack update (Supervision version)
        if len(veh_dets) > 0:
            detections = sv.Detections(
                xyxy=veh_dets,
                confidence=veh_confs,
                class_id=veh_cls,
            )
        
            detections = tracker.update_with_detections(detections)
        
            tracks = []

            print(f"\nFRAME {frame_no}")
            print(f"Vehicles detected: {len(tracks)}")
        
            for i in range(len(detections)):
                tid = detections.tracker_id[i]
        
                if tid is None:
                    continue
        
                x1, y1, x2, y2 = detections.xyxy[i]
                conf = float(detections.confidence[i])
                cls = int(detections.class_id[i])
        
                tracks.append([
                    x1,
                    y1,
                    x2,
                    y2,
                    int(tid),
                    conf,
                    cls
                ])
        else:
            tracks = []
             
        # 6. Dual-ROI violation logic
        frame_violations = set()
        
        for track in tracks:
            # track = [x1, y1, x2, y2, track_id, conf, cls]
            tx1, ty1, tx2, ty2 = track[0], track[1], track[2], track[3]
            tid = int(track[4])
            v_conf = float(track[5])
            v_class = VEHICLE_CLASSES.get(int(track[6]), "vehicle")
        
            cx = int((tx1 + tx2) / 2)
            cy = int((ty1 + ty2) / 2)
        
            in_a = point_in_roi(cx, cy, rois["A"])
            in_b = point_in_roi(cx, cy, rois["B"])


            print(
                f"Frame={frame_no} "
                f"ID={tid} "
                f"Class={v_class} "
                f"Center=({cx},{cy}) "
                f"inA={in_a} "
                f"inB={in_b}"
            )

            # If light is red and vehicle enters ROI-A → start watching
            if tl_state == "red" and in_a and tid not in violated:
                in_roi_a.add(tid)
                all_metadata[tid] = {
                    "vehicle_class": v_class,
                    "vehicle_conf": v_conf,
                    "bbox": [tx1, ty1, tx2, ty2],
                }
        
            # If watched vehicle reaches ROI-B → VIOLATION
            if tl_state == "red" and in_a and tid not in violated:
                violated.add(tid)
                vio_count += 1
        
                vid = f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_{tid}"
                meta = all_metadata.get(tid, {})
        
                plate = read_plate(frame, [tx1, ty1, tx2, ty2])
        
                v = Violation(
                    violation_id=vid,
                    timestamp=datetime.now().isoformat(),
                    frame_number=frame_no,
                    track_id=tid,
                    vehicle_class=meta.get("vehicle_class", v_class),
                    tl_state=tl_state,
                    tl_confidence=tl_conf,
                    vehicle_conf=meta.get("vehicle_conf", v_conf),
                    bbox_xyxy=meta.get("bbox", [tx1, ty1, tx2, ty2]),
                    license_plate=plate,
                )
        
                frame_copy = frame.copy()
                frame_copy = draw_rois(frame_copy, rois)
                frame_copy = draw_vehicle(
                    frame_copy,
                    v.bbox_xyxy,
                    tid,
                    v.vehicle_class,
                    is_violator=True,
                )
        
                draw_tl_state(frame_copy, tl_state, tl_conf)
        
                saved_path = save_violation_frame(frame_copy, v)
                v.frame_saved_at = saved_path
        
                violations.append(v)
                frame_violations.add(tid)
        
                print(
                    f"  🚨 VIOLATION #{vio_count}  "
                    f"track={tid}  plate={plate}  frame={frame_no}"
                )
 
        # 7. Draw annotations on frame
        frame = draw_rois(frame, rois)
        
        for track in tracks:
            # track = [x1, y1, x2, y2, track_id, conf, cls]
            tx1, ty1, tx2, ty2 = track[0], track[1], track[2], track[3]
            tid = int(track[4])
            v_cls = VEHICLE_CLASSES.get(int(track[6]), "vehicle")
        
            frame = draw_vehicle(
                frame,
                [tx1, ty1, tx2, ty2],
                tid,
                v_cls,
                is_violator=(tid in frame_violations)
            )
        
        draw_tl_state(frame, tl_state, tl_conf)
        
        elapsed = time.time() - t0 or 1e-9
        live_fps = frame_no / elapsed
        draw_hud(frame, frame_no, vio_count, live_fps)
        
        if writer:
            writer.write(frame)
        
        if frame_no % 50 == 0:
            print(
                f"  Frame {frame_no:5d}  |  "
                f"TL={tl_state:<8}  |  "
                f"violations so far={vio_count}"
            )
        
    if writer:
        writer.release()
    if cap:
        cap.release()
        
    print(f"\n✅ Done.  Total violations detected: {vio_count}")
    return violations

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2860998985.py, line 32)

In [15]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)
    for f in filenames:
        print("   ", f)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/kushagragupta28xoxo
/kaggle/input/datasets/kushagragupta28xoxo/traffic-light-violation
/kaggle/input/datasets/kushagragupta28xoxo/traffic-light-violation/test_images
    4.png
    1.png
    2.png
    3.png
/kaggle/input/datasets/kushagragupta28xoxo/traffic-video
    Test video 1.mp4


In [16]:
# =============================================================================
# CELL 6 — Run + generate reports (VIDEO VERSION)
# =============================================================================

SOURCE = "/kaggle/input/datasets/kushagragupta28xoxo/traffic-video/Test video 1.mp4"

# ── Run detection ─────────────────────────────────────────────────────────────
violations = run_violation_detection(
    source=SOURCE,

    # Adjust later if ROI preview looks wrong
    roi_a_frac=(0.60, 0.75)
    roi_b_frac=(0.46, 0.60)

    save_video=True,
    max_frames=None,      # process complete video
)

# ── Save JSON metadata ────────────────────────────────────────────────────────
json_path = REPORT_DIR / "violations.json"

with open(json_path, "w") as f:
    json.dump(
        [asdict(v) for v in violations],
        f,
        indent=2,
        default=float
    )

print(f"📄 JSON report → {json_path}")

# ── Generate human-readable report ────────────────────────────────────────────
txt_path = REPORT_DIR / "violation_report.txt"

with open(txt_path, "w") as f:

    f.write("=" * 65 + "\n")
    f.write("  TRAFFIC VIOLATION REPORT\n")
    f.write(f"  Generated : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"  Source    : {SOURCE}\n")
    f.write(f"  Total     : {len(violations)} violation(s)\n")
    f.write("=" * 65 + "\n\n")

    for i, v in enumerate(violations, 1):

        f.write(f"[{i:03d}]  Violation ID : {v.violation_id}\n")
        f.write(f"       Timestamp    : {v.timestamp}\n")
        f.write(f"       Frame        : {v.frame_number}\n")
        f.write(f"       Track ID     : {v.track_id}\n")
        f.write(f"       Vehicle type : {v.vehicle_class}\n")
        f.write(f"       License plate: {v.license_plate}\n")
        f.write(f"       TL state     : {v.tl_state} (conf={v.tl_confidence:.0%})\n")
        f.write(f"       Vehicle conf : {v.vehicle_conf:.0%}\n")
        f.write(f"       Evidence     : {v.frame_saved_at}\n")
        f.write("-" * 65 + "\n")

print(f"📄 Text report → {txt_path}")

# ── Summary statistics ────────────────────────────────────────────────────────
from collections import Counter

if violations:

    cls_counts = Counter(v.vehicle_class for v in violations)

    plate_detected = sum(
        1 for v in violations
        if v.license_plate != "undetected"
    )

    print("\n" + "=" * 45)
    print("  SUMMARY")
    print("=" * 45)

    print(f"  Total violations  : {len(violations)}")
    print(f"  Plates detected   : {plate_detected}/{len(violations)}")

    print("\n  By vehicle type:")

    for cls, cnt in cls_counts.most_common():
        print(f"      {cls:<15} {cnt}")

    print(f"\n  Evidence frames → {FRAMES_DIR}")
    print(f"  Annotated video → {VIDEO_DIR}")
    print(f"  Reports         → {REPORT_DIR}")

else:
    print("No violations detected.")

📹 Video  1280×720  30.0 fps
Frame 0: Traffic lights found = 5

FRAME 0
Vehicles detected: 0
Frame=0 ID=1 Class=car Center=(586,162) inA=False inB=False
Frame=0 ID=2 Class=car Center=(486,161) inA=False inB=False
Frame=0 ID=3 Class=car Center=(716,174) inA=False inB=False
Frame=0 ID=4 Class=car Center=(692,154) inA=False inB=False
Frame=0 ID=5 Class=car Center=(771,151) inA=False inB=False
Frame=0 ID=6 Class=car Center=(916,129) inA=False inB=False
Frame=0 ID=7 Class=car Center=(808,129) inA=False inB=False
Frame=0 ID=8 Class=car Center=(630,150) inA=False inB=False
Frame=0 ID=9 Class=car Center=(992,112) inA=False inB=False
Frame=0 ID=10 Class=car Center=(1031,109) inA=False inB=False
Frame=0 ID=11 Class=car Center=(543,154) inA=False inB=False
Frame=0 ID=12 Class=car Center=(749,137) inA=False inB=False
  Frame     0  |  TL=red       |  violations so far=0
Frame 1: Traffic lights found = 5

FRAME 1
Vehicles detected: 0
Frame=1 ID=1 Class=car Center=(586,162) inA=False inB=False
Frame=

In [ ]:
# =============================================================================
# CELL 7 — Quick visual check (display last 3 violation frames inline)
# =============================================================================
 
from IPython.display import display, Image as IPImage
import glob as _glob
 
saved_frames = sorted(_glob.glob(str(FRAMES_DIR / "*.jpg")))
if saved_frames:
    print(f"Showing last {min(3, len(saved_frames))} violation frame(s):\n")
    for fpath in saved_frames[-3:]:
        display(IPImage(filename=fpath, width=700))
else:
    print("No violation frames saved yet. Run Cell 6 with a real source first.")